In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from collections import Counter
import re
from PIL import Image
import ast

# Using kagglehub to download the dataset
import kagglehub
print("Downloading dataset...")
path = kagglehub.dataset_download("pes12017000148/food-ingredients-and-recipe-dataset-with-images")
print("Path to dataset files:", path)

# Ustawienie ścieżek do danych
data_path = os.path.join(path, 'Food Ingredients and Recipe Dataset with Image Name Mapping.csv')
images_path = os.path.join(path, 'Food Images', 'Food Images')

# Wczytanie danych
df = pd.read_csv(data_path)

# Sprawdzenie podstawowych informacji o zbiorze danych
print(f"Liczba rekordów: {len(df)}")
print(f"Kolumny: {df.columns.tolist()}")
print(df.head())

# Sprawdzenie brakujących wartości
print("\nBrakujące wartości:")
print(df.isnull().sum())


In [5]:
# Utworzenie przykładowej kolumny 'Cuisine' (jeśli nie istnieje), aby można było kontynuować trening
import random

if 'Cuisine' not in df.columns:
    print("Kolumna 'Cuisine' nie istnieje. Tworzenie uproszczonej kolumny Cuisine na potrzeby demonstracyjne...")
    # Lista popularnych kuchni
    cuisines = ['Italian', 'Mexican', 'Asian', 'American', 'Indian', 'Mediterranean']
    # Przypiszemy je losowo, w realnym projekcie można by użyć słów kluczowych ze składników lub nazw potraw
    # Alternatywnie: naiwne przypisanie na podstawie słów kluczowych:
    def assign_cuisine(title):
        title = str(title).lower()
        if any(word in title for word in ['pasta', 'pizza', 'italian', 'lasagna']):
            return 'Italian'
        if any(word in title for word in ['taco', 'mexican', 'enchilada', 'burrito']):
            return 'Mexican'
        if any(word in title for word in ['curry', 'indian', 'tikka', 'masala']):
            return 'Indian'
        if any(word in title for word in ['soy', 'asian', 'stir-fry', 'chinese', 'japanese']):
            return 'Asian'
        return 'American'
        
    df['Cuisine'] = df['Title'].apply(assign_cuisine)

cuisine_counts = df['Cuisine'].value_counts()
print("\nLiczność klas kuchni:")
print(cuisine_counts)

# Wizualizacja liczności klas kuchni
plt.figure(figsize=(10, 5))
sns.barplot(x=cuisine_counts.index, y=cuisine_counts.values)
plt.title('Liczność klas kuchni')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# Analiza składników
df['Cleaned_Ingredients'] = df['Cleaned_Ingredients'].apply(ast.literal_eval)

all_ingredients = []
for ingredients_list in df['Cleaned_Ingredients']:
    all_ingredients.extend(ingredients_list)

ingredient_counts = Counter(all_ingredients)
most_common_ingredients = ingredient_counts.most_common(20)

print("\nNajpopularniejsze składniki:")
for ingredient, count in most_common_ingredients:
    print(f"{ingredient}: {count}")

# Analiza brakujących obrazów i czyszczenie DataFrame
missing_images = []
valid_indices = []

for idx, row in df.iterrows():
    image_name = row['Image_Name']
    if pd.isna(image_name):
        missing_images.append(idx)
        continue
    image_path = os.path.join(images_path, str(image_name) + '.jpg')
    if os.path.exists(image_path):
        valid_indices.append(idx)
    else:
        missing_images.append(idx)

print(f"\nLiczba brakujących obrazów: {len(missing_images)}")

# Usunięcie wierszy z brakującymi obrazami
df = df.loc[valid_indices].reset_index(drop=True)
print(f"Liczba wierszy po usunięciu brakujących obrazów: {len(df)}")


In [7]:
import nltk
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Pobieranie stopwords
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

# Funkcja do przetwarzania składników
def preprocess_ingredients(ingredients_list):
    # Łączenie składników w jeden string
    ingredients_text = ' '.join(ingredients_list)
    # Usuwanie liczb, jednostek miary itp.
    ingredients_text = re.sub(r'\d+', '', ingredients_text)
    ingredients_text = re.sub(r'cup|tablespoon|teaspoon|pound|ounce|tbsp|tsp|oz|lb|g|ml|l', '', ingredients_text)
    # Usuwanie znaków interpunkcyjnych
    ingredients_text = re.sub(r'[^\w\s]', '', ingredients_text)
    # Konwersja na małe litery
    ingredients_text = ingredients_text.lower()
    # Usuwanie stopwords
    words = ingredients_text.split()
    filtered_words = [word for word in words if word not in stop_words]
    return ' '.join(filtered_words)

# Przetwarzanie składników
df['Processed_Ingredients'] = df['Cleaned_Ingredients'].apply(preprocess_ingredients)

# Metoda 1: TF-IDF
tfidf_vectorizer = TfidfVectorizer(max_features=1000)
X_ingredients_tfidf = tfidf_vectorizer.fit_transform(df['Processed_Ingredients'])

# Metoda 2: Tokenizacja i sekwencjonowanie dla embeddings
tokenizer = Tokenizer(num_words=5000)
tokenizer.fit_on_texts(df['Processed_Ingredients'])
X_ingredients_seq = tokenizer.texts_to_sequences(df['Processed_Ingredients'])
X_ingredients_padded = pad_sequences(X_ingredients_seq, maxlen=100)


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\MAXIMUS_PC\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


In [8]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.resnet50 import preprocess_input as resnet_preprocess
from tensorflow.keras.applications.vgg16 import preprocess_input as vgg_preprocess
import os
import pandas as pd

# Konfiguracja parametrów obrazów i ścieżek
BATCH_SIZE = 32
IMAGE_SIZE = (224, 224)

# Zapisanie pełnych ścieżek do obrazów w DataFrame
df['Image_Path'] = df['Image_Name'].apply(lambda x: os.path.join(images_path, str(x) + '.jpg'))

# Funkcja do wczytywania i mapowania obrazów z tf.data.Dataset
def load_and_preprocess_image(image_path, label):
    img = tf.io.read_file(image_path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, IMAGE_SIZE)
    # Zastosowanie pre-processingu ResNet (jako domyślnego dla naszych modeli ResNet)
    img = tf.keras.applications.resnet50.preprocess_input(img)
    return img, label

def load_and_preprocess_vgg_image(image_path, label):
    img = tf.io.read_file(image_path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, IMAGE_SIZE)
    # Zastosowanie pre-processingu VGG (dla modelu VGG)
    img = tf.keras.applications.vgg16.preprocess_input(img)
    return img, label

print("Stworzono funkcje ładujące obrazy do tf.data.Dataset. Unikniemy MemoryError!")


In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

# Kodowanie etykiet
label_encoder = LabelEncoder()
df['Cuisine_Label'] = label_encoder.fit_transform(df['Cuisine'])

# Podział danych na zbiory treningowy i testowy (indeksy)
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['Cuisine_Label'])

# Przygotowanie danych tekstowych
X_train_text_tfidf = X_ingredients_tfidf[train_df.index]
X_test_text_tfidf = X_ingredients_tfidf[test_df.index]

X_train_text_padded = X_ingredients_padded[train_df.index]
X_test_text_padded = X_ingredients_padded[test_df.index]

y_train = train_df['Cuisine_Label'].values
y_test = test_df['Cuisine_Label'].values

# Tworzenie tf.data.Dataset dla ResNet
train_dataset_resnet = tf.data.Dataset.from_tensor_slices((train_df['Image_Path'].values, y_train))
train_dataset_resnet = train_dataset_resnet.map(load_and_preprocess_image, num_parallel_calls=tf.data.AUTOTUNE)
train_dataset_resnet = train_dataset_resnet.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

test_dataset_resnet = tf.data.Dataset.from_tensor_slices((test_df['Image_Path'].values, y_test))
test_dataset_resnet = test_dataset_resnet.map(load_and_preprocess_image, num_parallel_calls=tf.data.AUTOTUNE)
test_dataset_resnet = test_dataset_resnet.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

# Tworzenie tf.data.Dataset dla VGG
train_dataset_vgg = tf.data.Dataset.from_tensor_slices((train_df['Image_Path'].values, y_train))
train_dataset_vgg = train_dataset_vgg.map(load_and_preprocess_vgg_image, num_parallel_calls=tf.data.AUTOTUNE)
train_dataset_vgg = train_dataset_vgg.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

test_dataset_vgg = tf.data.Dataset.from_tensor_slices((test_df['Image_Path'].values, y_test))
test_dataset_vgg = test_dataset_vgg.map(load_and_preprocess_vgg_image, num_parallel_calls=tf.data.AUTOTUNE)
test_dataset_vgg = test_dataset_vgg.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

print(f"Dane podzielone. Trening: {len(train_df)}, Test: {len(test_df)}")


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Embedding, Flatten, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

# Model MLP dla składników (TF-IDF)
def create_mlp_model(input_dim, num_classes):
    model = Sequential([
        Input(shape=(input_dim,)),
        Dense(512, activation='relu'),
        Dropout(0.3),
        Dense(256, activation='relu'),
        Dropout(0.3),
        Dense(128, activation='relu'),
        Dense(num_classes, activation='softmax')
    ])
    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

# Model dla TF-IDF
mlp_tfidf_model = create_mlp_model(X_train_text_tfidf.shape[1], len(label_encoder.classes_))

# Trenowanie modelu
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

# Przekonwertowanie sparse matrix TF-IDF na dense array
X_train_dense = X_train_text_tfidf.toarray()
X_test_dense = X_test_text_tfidf.toarray()

print("Trenowanie modelu MLP (TF-IDF)...")
history_mlp_tfidf = mlp_tfidf_model.fit(
    X_train_dense, y_train,
    validation_data=(X_test_dense, y_test),
    epochs=20,
    batch_size=32,
    callbacks=[early_stopping],
    verbose=1
)

# Model dla embeddings
def create_embedding_model(vocab_size, embedding_dim, input_length, num_classes):
    model = Sequential([
        Input(shape=(input_length,)),
        Embedding(vocab_size, embedding_dim),
        Flatten(),
        Dense(256, activation='relu'),
        Dropout(0.3),
        Dense(128, activation='relu'),
        Dense(num_classes, activation='softmax')
    ])
    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

# Parametry modelu
vocab_size = len(tokenizer.word_index) + 1
embedding_dim = 100
input_length = X_ingredients_padded.shape[1]

# Tworzenie i trenowanie modelu Embeddings
mlp_embedding_model = create_embedding_model(vocab_size, embedding_dim, input_length, len(label_encoder.classes_))

print("\nTrenowanie modelu MLP (Embeddings)...")
history_mlp_embedding = mlp_embedding_model.fit(
    X_train_text_padded, y_train,
    validation_data=(X_test_text_padded, y_test),
    epochs=20,
    batch_size=32,
    callbacks=[early_stopping],
    verbose=1
)


In [ ]:
from tensorflow.keras.applications import ResNet50, VGG16
from tensorflow.keras.layers import GlobalAveragePooling2D

# Model ResNet50
def create_resnet_model(num_classes):
    base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
    
    # Zamrożenie warstw bazowego modelu
    for layer in base_model.layers:
        layer.trainable = False
    
    model = Sequential([
        Input(shape=(224, 224, 3)),
        base_model,
        GlobalAveragePooling2D(),
        Dense(512, activation='relu'),
        Dropout(0.3),
        Dense(num_classes, activation='softmax')
    ])
    
    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

# Tworzenie i trenowanie modelu ResNet
resnet_model = create_resnet_model(len(label_encoder.classes_))

print("Trenowanie modelu ResNet50...")
history_resnet = resnet_model.fit(
    train_dataset_resnet,
    validation_data=test_dataset_resnet,
    epochs=10, # Zmniejszono liczbę epok dla przyspieszenia ewaluacji
    callbacks=[early_stopping],
    verbose=1
)

# Model VGG16
def create_vgg_model(num_classes):
    base_model = VGG16(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
    
    # Zamrożenie warstw bazowego modelu
    for layer in base_model.layers:
        layer.trainable = False
    
    model = Sequential([
        Input(shape=(224, 224, 3)),
        base_model,
        GlobalAveragePooling2D(),
        Dense(512, activation='relu'),
        Dropout(0.3),
        Dense(num_classes, activation='softmax')
    ])
    
    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

# Tworzenie i trenowanie modelu VGG
vgg_model = create_vgg_model(len(label_encoder.classes_))

print("\nTrenowanie modelu VGG16...")
history_vgg = vgg_model.fit(
    train_dataset_vgg,
    validation_data=test_dataset_vgg,
    epochs=10,
    callbacks=[early_stopping],
    verbose=1
)


In [ ]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import concatenate

# Przygotowanie tf.data.Dataset dla modelu multimodalnego
def multimodal_generator(df, text_features, labels, batch_size):
    # text_features must be dense array
    text_dataset = tf.data.Dataset.from_tensor_slices(text_features)
    
    # image dataset
    img_dataset = tf.data.Dataset.from_tensor_slices(df['Image_Path'].values)
    
    def load_img(path):
        img = tf.io.read_file(path)
        img = tf.image.decode_jpeg(img, channels=3)
        img = tf.image.resize(img, IMAGE_SIZE)
        img = tf.keras.applications.resnet50.preprocess_input(img)
        return img
        
    img_dataset = img_dataset.map(load_img, num_parallel_calls=tf.data.AUTOTUNE)
    
    # label dataset
    label_dataset = tf.data.Dataset.from_tensor_slices(labels)
    
    # zip them together into a dict: ({"text_input": text, "image_input": image}, label)
    dataset = tf.data.Dataset.zip(((text_dataset, img_dataset), label_dataset))
    dataset = dataset.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    
    return dataset

# Zbudowanie zestawów danych multimodalnych
train_multimodal_dataset = multimodal_generator(train_df, X_train_dense, y_train, BATCH_SIZE)
test_multimodal_dataset = multimodal_generator(test_df, X_test_dense, y_test, BATCH_SIZE)

# Funkcja do tworzenia modelu multimodalnego
def create_multimodal_model(text_input_dim, image_input_shape, num_classes):
    # Gałąź tekstowa (MLP)
    text_input = Input(shape=(text_input_dim,), name="text_input")
    text_features = Dense(256, activation='relu')(text_input)
    text_features = Dropout(0.3)(text_features)
    text_features = Dense(128, activation='relu')(text_features)
    
    # Gałąź obrazowa (CNN)
    image_input = Input(shape=image_input_shape, name="image_input")
    base_model = ResNet50(weights='imagenet', include_top=False, input_tensor=image_input)
    
    # Zamrożenie warstw bazowego modelu
    for layer in base_model.layers:
        layer.trainable = False
    
    image_features = GlobalAveragePooling2D()(base_model.output)
    image_features = Dense(256, activation='relu')(image_features)
    image_features = Dropout(0.3)(image_features)
    
    # Połączenie gałęzi
    combined = concatenate([text_features, image_features])
    
    # Wspólne warstwy
    x = Dense(256, activation='relu')(combined)
    x = Dropout(0.3)(x)
    x = Dense(128, activation='relu')(x)
    
    # Warstwa wyjściowa
    output = Dense(num_classes, activation='softmax')(x)
    
    # Tworzenie modelu
    model = Model(inputs=[text_input, image_input], outputs=output)
    
    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    
    return model

# Tworzenie modelu multimodalnego
multimodal_model = create_multimodal_model(
    X_train_dense.shape[1], 
    (224, 224, 3), 
    len(label_encoder.classes_)
)

print("Trenowanie modelu multimodalnego...")
# Trenowanie modelu
history_multimodal = multimodal_model.fit(
    train_multimodal_dataset,
    validation_data=test_multimodal_dataset,
    epochs=10,
    callbacks=[early_stopping],
    verbose=1
)


In [ ]:
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
import numpy as np

# Ewaluacja modelu składników (MLP)
y_pred_mlp = mlp_tfidf_model.predict(X_test_dense)
y_pred_mlp_classes = np.argmax(y_pred_mlp, axis=1)

print("\nWyniki modelu MLP (składniki):")
print(f"Accuracy: {accuracy_score(y_test, y_pred_mlp_classes):.4f}")
print(f"F1 Score (weighted): {f1_score(y_test, y_pred_mlp_classes, average='weighted'):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_mlp_classes, target_names=label_encoder.classes_))

# Aby użyć predict z tf.data.Dataset:
y_pred_resnet = resnet_model.predict(test_dataset_resnet)
y_pred_resnet_classes = np.argmax(y_pred_resnet, axis=1)

print("\nWyniki modelu ResNet (obrazy):")
print(f"Accuracy: {accuracy_score(y_test, y_pred_resnet_classes):.4f}")
print(f"F1 Score (weighted): {f1_score(y_test, y_pred_resnet_classes, average='weighted'):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_resnet_classes, target_names=label_encoder.classes_))

# Ewaluacja modelu obrazów (VGG)
y_pred_vgg = vgg_model.predict(test_dataset_vgg)
y_pred_vgg_classes = np.argmax(y_pred_vgg, axis=1)

print("\nWyniki modelu VGG (obrazy):")
print(f"Accuracy: {accuracy_score(y_test, y_pred_vgg_classes):.4f}")
print(f"F1 Score (weighted): {f1_score(y_test, y_pred_vgg_classes, average='weighted'):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_vgg_classes, target_names=label_encoder.classes_))

# Ewaluacja modelu multimodalnego
y_pred_multimodal = multimodal_model.predict(test_multimodal_dataset)
y_pred_multimodal_classes = np.argmax(y_pred_multimodal, axis=1)

print("\nWyniki modelu multimodalnego:")
print(f"Accuracy: {accuracy_score(y_test, y_pred_multimodal_classes):.4f}")
print(f"F1 Score (weighted): {f1_score(y_test, y_pred_multimodal_classes, average='weighted'):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_multimodal_classes, target_names=label_encoder.classes_))


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Funkcja do wizualizacji macierzy pomyłek
def plot_confusion_matrix(y_true, y_pred, class_names, title):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(12, 10))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
    plt.title(title)
    plt.ylabel('Rzeczywista klasa')
    plt.xlabel('Przewidywana klasa')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

# Wizualizacja macierzy pomyłek dla wszystkich modeli
plot_confusion_matrix(y_test, y_pred_mlp_classes, label_encoder.classes_, 'Macierz pomyłek - Model MLP (składniki)')
plot_confusion_matrix(y_test, y_pred_resnet_classes, label_encoder.classes_, 'Macierz pomyłek - Model ResNet (obrazy)')
plot_confusion_matrix(y_test, y_pred_vgg_classes, label_encoder.classes_, 'Macierz pomyłek - Model VGG (obrazy)')
plot_confusion_matrix(y_test, y_pred_multimodal_classes, label_encoder.classes_, 'Macierz pomyłek - Model multimodalny')


In [ ]:
# Funkcja do wizualizacji historii treningu
def plot_training_history(history, title):
    plt.figure(figsize=(12, 5))
    
    # Wykres funkcji straty
    plt.subplot(1, 2, 1)
    plt.plot(history.history['loss'], label='Trening')
    plt.plot(history.history['val_loss'], label='Walidacja')
    plt.title(f'{title} - Funkcja straty')
    plt.xlabel('Epoka')
    plt.ylabel('Strata')
    plt.legend()
    
    # Wykres dokładności
    plt.subplot(1, 2, 2)
    plt.plot(history.history['accuracy'], label='Trening')
    plt.plot(history.history['val_accuracy'], label='Walidacja')
    plt.title(f'{title} - Dokładność')
    plt.xlabel('Epoka')
    plt.ylabel('Dokładność')
    plt.legend()
    
    plt.tight_layout()
    plt.show()

# Wizualizacja historii treningu dla wszystkich modeli
plot_training_history(history_mlp_tfidf, 'Model MLP (składniki)')
plot_training_history(history_resnet, 'Model ResNet (obrazy)')
plot_training_history(history_vgg, 'Model VGG (obrazy)')
plot_training_history(history_multimodal, 'Model multimodalny')


In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import cv2
import os

# Implementacja Grad-CAM
def make_gradcam_heatmap(img_array, model, last_conv_layer_name, pred_index=None):
    # Tworzenie modelu mapującego obraz wejściowy na aktywacje ostatniej warstwy
    grad_model = tf.keras.models.Model(
        model.inputs, 
        [model.get_layer(last_conv_layer_name).output, model.output]
    )
    
    with tf.GradientTape() as tape:
        last_conv_layer_output, preds = grad_model(img_array)
        if pred_index is None:
            pred_index = tf.argmax(preds[0])
        class_channel = preds[:, pred_index]
    
    grads = tape.gradient(class_channel, last_conv_layer_output)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    
    last_conv_layer_output = last_conv_layer_output[0]
    heatmap = last_conv_layer_output @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / tf.math.reduce_max(heatmap)
    return heatmap.numpy()

# Funkcja do wizualizacji Grad-CAM
def display_gradcam(image_path, model, last_conv_layer_name, pred_index=None, alpha=0.4):
    img = tf.keras.preprocessing.image.load_img(image_path, target_size=(224, 224))
    img_array = tf.keras.preprocessing.image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)
    img_array = tf.keras.applications.resnet50.preprocess_input(img_array)
    
    heatmap = make_gradcam_heatmap(img_array, model, last_conv_layer_name, pred_index)
    
    img = cv2.imread(image_path)
    img = cv2.resize(img, (224, 224))
    heatmap = cv2.resize(heatmap, (img.shape[1], img.shape[0]))
    heatmap = np.uint8(255 * heatmap)
    heatmap = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)
    
    superimposed_img = heatmap * alpha + img
    superimposed_img = np.clip(superimposed_img, 0, 255).astype('uint8')
    
    plt.figure(figsize=(12, 4))
    plt.subplot(1, 3, 1)
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title('Oryginalny obraz')
    plt.axis('off')
    
    plt.subplot(1, 3, 2)
    plt.imshow(cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB))
    plt.title('Mapa cieplna Grad-CAM')
    plt.axis('off')
    
    plt.subplot(1, 3, 3)
    plt.imshow(cv2.cvtColor(superimposed_img, cv2.COLOR_BGR2RGB))
    plt.title('Nałożona mapa cieplna')
    plt.axis('off')
    
    plt.tight_layout()
    plt.show()

# Przykładowa wizualizacja Grad-CAM dla kilku obrazów testowych
sample_indices = np.random.choice(len(test_df), min(3, len(test_df)), replace=False)
for idx in sample_indices:
    row = test_df.iloc[idx]
    image_path = row['Image_Path']
    pred_class = y_pred_resnet_classes[idx]
    true_class = row['Cuisine_Label']
    
    print(f"Obraz: {row['Title']}")
    print(f"Prawdziwa klasa: {label_encoder.classes_[true_class]}")
    print(f"Przewidywana klasa: {label_encoder.classes_[pred_class]}")
    
    display_gradcam(
        image_path, 
        resnet_model, 
        'conv5_block3_out',  # Ostatnia warstwa konwolucyjna w ResNet50
        pred_class
    )


In [ ]:
# Porównanie dokładności modeli
models = ['MLP (składniki)', 'ResNet (obrazy)', 'VGG (obrazy)', 'Multimodalny']
accuracies = [
    accuracy_score(y_test, y_pred_mlp_classes),
    accuracy_score(y_test, y_pred_resnet_classes),
    accuracy_score(y_test, y_pred_vgg_classes),
    accuracy_score(y_test, y_pred_multimodal_classes)
]
f1_scores = [
    f1_score(y_test, y_pred_mlp_classes, average='weighted'),
    f1_score(y_test, y_pred_resnet_classes, average='weighted'),
    f1_score(y_test, y_pred_vgg_classes, average='weighted'),
    f1_score(y_test, y_pred_multimodal_classes, average='weighted')
]

# Wizualizacja porównania modeli
plt.figure(figsize=(12, 6))

plt.subplot(1, 2, 1)
plt.bar(models, accuracies, color=['blue', 'green', 'orange', 'red'])
plt.title('Porównanie dokładności modeli')
plt.ylabel('Accuracy')
plt.ylim(0, 1)
plt.xticks(rotation=45, ha='right')

plt.subplot(1, 2, 2)
plt.bar(models, f1_scores, color=['blue', 'green', 'orange', 'red'])
plt.title('Porównanie F1-Score modeli')
plt.ylabel('F1-Score (weighted)')
plt.ylim(0, 1)
plt.xticks(rotation=45, ha='right')

plt.tight_layout()
plt.show()

# Tabela porównawcza
comparison_df = pd.DataFrame({
    'Model': models,
    'Accuracy': accuracies,
    'F1-Score': f1_scores
})
print("\nTabela porównawcza modeli:")
print(comparison_df)
